# Lesson 4: Seq2Seq Models for Question Answering

## Overview
In this exercise, you'll build a seq2seq model that answers questions based on provided context. The model uses:
- **Encoder**: Bidirectional LSTM that compresses context into a fixed-size vector (THE BOTTLENECK)
- **Decoder**: Unidirectional LSTM that generates answers token-by-token
- **Teacher Forcing**: During training, the model uses ground truth tokens instead of predictions

The key insight: ALL context (whether 10 or 70 words) gets compressed into the same 256D vector. This creates a bottleneck that hurts performance on longer contexts. We'll measure this empirically!

## Setup: Import Required Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import random
from tqdm import tqdm

# Import custom modules
from data import (
    SyntheticQAGenerator,
    build_vocabulary,
    encode_text,
    decode_text,
    PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN, SEP_TOKEN
)
from models import Encoder, Decoder, Seq2Seq, count_parameters

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## Part A: Data Generation and Vocabulary

### A.1: Generate Synthetic Q&A Data
Create synthetic question-answer pairs with controlled context lengths to clearly demonstrate the bottleneck effect.

In [4]:
print("="*80)
print("GENERATING Q&A DATASET")
print("="*80)


generator = SyntheticQAGenerator()


qa_data, context_lengths = generator.generate_dataset(
    n_short=400,   # 8-12 words
    n_medium=400,  # 25-35 words
    n_long=400,    # 50-70 words
    shuffle=True
)

print(f"\n✓ Generated {len(qa_data)} Q&A pairs")
print(f"\nContext Length Distribution:")
print(f"  Short (<15 words):   {sum(1 for l in context_lengths if l < 15):3d} examples")
print(f"  Medium (15-45 words): {sum(1 for l in context_lengths if 15 <= l < 45):3d} examples")
print(f"  Long (45+ words):     {sum(1 for l in context_lengths if l >= 45):3d} examples")

# Show examples
print("\n" + "="*80)
print("EXAMPLE Q&A PAIRS")
print("="*80)

for label, min_len, max_len in [('SHORT', 0, 15), ('MEDIUM', 15, 45), ('LONG', 45, 100)]:
    for ctx, q, ans in qa_data:
        ctx_len = len(ctx.split())
        if min_len <= ctx_len < max_len:
            print(f"\n{label} Context ({ctx_len} words):")
            print(f"  Context: {ctx}")
            print(f"  Question: {q}")
            print(f"  Answer: {ans}")
            break

GENERATING Q&A DATASET

✓ Generated 1200 Q&A pairs

Context Length Distribution:
  Short (<15 words):   400 examples
  Medium (15-45 words): 726 examples
  Long (45+ words):      74 examples

EXAMPLE Q&A PAIRS

SHORT Context (6 words):
  Context: the document is in the clinic
  Question: where is the document
  Answer: in the clinic

MEDIUM Context (17 words):
  Context: Sam who is narrow and rather certainly clearly actually obviously surely lives in London and works there
  Question: where does Sam live
  Answer: London

LONG Context (48 words):
  Context: Charlie who is light and cold and absolutely fully completely actually definitely totally quite certainly just still now that I think about it lives in the city of Sydney which is still also definitely completely clearly actually somewhat obviously and works at the warehouse there every single day
  Question: where does Charlie live
  Answer: Sydney


### A.2: Build Vocabulary
Create a vocabulary from all contexts, questions, and answers.

In [5]:

print("Building vocabulary...")
vocab, idx2word = build_vocabulary(qa_data)
vocab_size = len(vocab)

print(f"\nVocabulary size: {vocab_size}")
print(f"Sample words: {list(vocab.keys())[5:20]}")

# Test encoding/decoding
test_text = "Alice lives in Paris"
encoded = encode_text(test_text, vocab)
decoded = decode_text(encoded, idx2word)

print(f"\nTest encoding:")
print(f"  Original: {test_text}")
print(f"  Encoded:  {encoded}")
print(f"  Decoded:  {decoded}")

Building vocabulary...

Vocabulary size: 220
Sample words: ['sam', 'who', 'is', 'narrow', 'and', 'rather', 'certainly', 'clearly', 'actually', 'obviously', 'surely', 'lives', 'in', 'london', 'works']

Test encoding:
  Original: Alice lives in Paris
  Encoded:  [89, 16, 17, 211]
  Decoded:  alice lives in paris


### A.3: Create PyTorch Dataset
Encode Q&A pairs into token sequences.

In [6]:
class QADataset(Dataset):
    """PyTorch Dataset for Q&A pairs."""
    
    def __init__(self, data, vocab, idx2word, max_len=100):
        self.data = data
        self.vocab = vocab
        self.idx2word = idx2word
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        ctx, q, ans = self.data[idx]
        
        # Encode
        ctx_ids = encode_text(ctx, self.vocab)
        q_ids = encode_text(q, self.vocab)
        ans_ids = encode_text(ans, self.vocab)
        
        # Encoder input: context + SEP + question + EOS
        enc_in = (ctx_ids + [SEP_TOKEN] + q_ids + [EOS_TOKEN])[:self.max_len]
        enc_in += [PAD_TOKEN] * (self.max_len - len(enc_in))
        
        # Decoder input: SOS + answer
        dec_in = ([SOS_TOKEN] + ans_ids)[:self.max_len]
        dec_in += [PAD_TOKEN] * (self.max_len - len(dec_in))
        
        # Decoder target: answer + EOS
        dec_out = (ans_ids + [EOS_TOKEN])[:self.max_len]
        dec_out += [PAD_TOKEN] * (self.max_len - len(dec_out))
        
        return {
            'encoder_input': torch.LongTensor(enc_in),
            'decoder_input': torch.LongTensor(dec_in),
            'decoder_target': torch.LongTensor(dec_out),
            'context_length': len(ctx_ids)
        }


def collate_fn(batch):
    """Collate function for DataLoader."""
    return {
        'encoder_input': torch.stack([item['encoder_input'] for item in batch]),
        'decoder_input': torch.stack([item['decoder_input'] for item in batch]),
        'decoder_target': torch.stack([item['decoder_target'] for item in batch]),
        'context_lengths': torch.tensor([item['context_length'] for item in batch])
    }

# Split data: 70% train, 20% val, 10% test
train_size = int(0.70 * len(qa_data))
val_size = int(0.20 * len(qa_data))

train_data = qa_data[:train_size]
val_data = qa_data[train_size:train_size+val_size]
test_data = qa_data[train_size+val_size:]

# Create datasets
train_dataset = QADataset(train_data, vocab, idx2word)
val_dataset = QADataset(val_data, vocab, idx2word)
test_dataset = QADataset(test_data, vocab, idx2word)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_dataset):4d} examples ({len(train_loader):2d} batches)")
print(f"Val:   {len(val_dataset):4d} examples ({len(val_loader):2d} batches)")
print(f"Test:  {len(test_dataset):4d} examples ({len(test_loader):2d} batches)")

Train:  840 examples (27 batches)
Val:    240 examples ( 8 batches)
Test:   120 examples ( 4 batches)


## Part B: Model Architecture

### B.1: Initialize Model
Create encoder, decoder, and seq2seq model. Make sure you've implemented `models.py` first!

In [8]:
print("="*80)
print("BUILDING SEQ2SEQ MODEL")
print("="*80)

# Model hyperparameters
EMBED_DIM = 128
HIDDEN_DIM = 256

# Create encoder and decoder (from models.py)
encoder = Encoder(vocab_size, EMBED_DIM, HIDDEN_DIM, padding_idx=PAD_TOKEN).to(device)
decoder = Decoder(vocab_size, EMBED_DIM, HIDDEN_DIM, padding_idx=PAD_TOKEN).to(device)
model = Seq2Seq(encoder, decoder).to(device)

num_params = count_parameters(model)

print(f"\n✓ Model created successfully!")
print(f"\nArchitecture:")
print(f"  Embedding dimension:  {EMBED_DIM}")
print(f"  Hidden dimension:     {HIDDEN_DIM}")
print(f"  Context vector size:  {HIDDEN_DIM} (THE BOTTLENECK!)")
print(f"  Vocabulary size:      {vocab_size}")
print(f"  Total parameters:     {num_params:,}")

print(f"\n⚠️  Key Point: ALL contexts (short or long) compress to {HIDDEN_DIM}D vector!")

BUILDING SEQ2SEQ MODEL

✓ Model created successfully!

Architecture:
  Embedding dimension:  128
  Hidden dimension:     256
  Context vector size:  256 (THE BOTTLENECK!)
  Vocabulary size:      220
  Total parameters:     1,561,308

⚠️  Key Point: ALL contexts (short or long) compress to 256D vector!


## Part C: Training

### C.1: Setup Optimizer and Loss

In [ ]:
# TODO: Setup training components
# Hint:
# 1. Choose a loss function appropriate for classification (CrossEntropyLoss) and ignore padding tokens
# 2. Choose an optimizer (Adam is a good choice) with a reasonable learning rate
# 3. Understand what train_epoch() and evaluate() should do:
#    - train_epoch: Use teacher forcing during training (decoder sees ground truth tokens)
#    - evaluate: No teacher forcing during validation/test (decoder only sees its own predictions)

### C.2: Train Model

In [9]:
# TODO: Train the model
# Hint:
# 1. Run for ~30 epochs and track both training and validation losses
# 2. Decay the teacher forcing ratio over time (start high, decrease gradually)
# 3. Use train_epoch() for training and evaluate() for validation after each epoch
# 4. Print metrics periodically to monitor progress (every 5 epochs)

# Training setup
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, dataloader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for batch in tqdm(dataloader, desc="Training", leave=False):
        src = batch['encoder_input'].to(device)
        trg_in = batch['decoder_input'].to(device)
        trg_out = batch['decoder_target'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass with teacher forcing
        output = model(src, trg_in, teacher_forcing_ratio=0.5)
        
        # Calculate loss
        output = output.reshape(-1, vocab_size)
        trg_out = trg_out.reshape(-1)
        loss = criterion(output, trg_out)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            src = batch['encoder_input'].to(device)
            trg_in = batch['decoder_input'].to(device)
            trg_out = batch['decoder_target'].to(device)
            
            output = model(src, trg_in, teacher_forcing_ratio=0)
            
            output = output.reshape(-1, vocab_size)
            trg_out = trg_out.reshape(-1)
            loss = criterion(output, trg_out)
            
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

# Train
NUM_EPOCHS = 30
train_losses = []
val_losses = []

print("="*80)
print("TRAINING (This will take a few minutes)")
print("="*80)
print("Epoch | Train Loss | Val Loss")
print("-" * 40)

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"{epoch+1:5d} | {train_loss:10.4f} | {val_loss:8.4f}")

print("\n✓ Training complete!")
print(f"\nFinal Results:")
print(f"  Train loss: {train_losses[0]:.3f} → {train_losses[-1]:.3f}")
print(f"  Val loss:   {val_losses[0]:.3f} → {val_losses[-1]:.3f}")
print(f"  Improvement: {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")

TRAINING (This will take a few minutes)
Epoch | Train Loss | Val Loss
----------------------------------------


    5 |     1.5356 |   1.5585


   10 |     1.3120 |   1.3495


   15 |     0.9239 |   1.0356


   20 |     0.7476 |   0.9437


   25 |     0.4850 |   0.6101


   30 |     0.3228 |   0.4395

✓ Training complete!

Final Results:
  Train loss: 3.426 → 0.323
  Val loss:   2.489 → 0.440
  Improvement: 90.6%


### C.3: Plot Training Curves

In [ ]:
# TODO: Visualize training progress
# Hint:
# 1. Plot training and validation losses on the same graph
# 2. Use proper labels, titles, and gridlines for clarity
# 3. Observe: Does training loss decrease? Does validation loss converge or diverge?
# 4. Look for signs of overfitting (training goes down but validation goes up)

## Part D: Inference and Bottleneck Analysis

### D.1: Inference Function
Generate answers for new context-question pairs.

In [ ]:
# TODO: Implement inference (answer generation)
# Hint:
# 1. Encode the context and question together (using your encode_text function)
# 2. Pass the encoded input through the encoder to get the context vector (the bottleneck!)
# 3. Use the context vector to seed the decoder
# 4. Autoregressively generate tokens one at a time:
#    - Take the current generated tokens as input
#    - Decoder outputs logits for the next token
#    - Select the token with highest probability (greedy decoding)
#    - Stop when you generate an <END> token or reach max_length
# 5. Decode the token indices back to text

# Test on some examples from the test set and compare predictions to ground truth

### D.2: Calculate Accuracy

In [ ]:
# TODO: Calculate accuracy on test set
# Hint:
# 1. Iterate through the test set
# 2. Generate predictions using your inference function
# 3. Compare each prediction to the ground truth answer (exact match or case-insensitive)
# 4. Calculate the percentage of correct predictions
# 5. Print the overall test accuracy

### D.3: Bottleneck Analysis

**THE KEY INSIGHT**: The encoder compresses all contexts into the same 256D vector, regardless of context length. This should hurt performance significantly on longer contexts.

Let's measure this empirically!

In [ ]:
# TODO: THE KEY EXERCISE - Analyze accuracy by context length
# Hint:
# 1. For each test example, collect: context_length (word count), prediction, ground_truth, is_correct
# 2. Bucket results into three categories:
#    - Short contexts: 8-12 words
#    - Medium contexts: 25-35 words
#    - Long contexts: 50-70 words
# 3. Calculate accuracy within each bucket
# 4. Plot a bar chart showing the three accuracies
# 5. OBSERVE: You should see 30-40% degradation from short to long
#    - Why? Because all contexts compress to THE SAME 256D vector!
# 6. This empirically demonstrates why attention mechanisms are needed (next lesson)

## Summary

### What We Learned

1. **The Bottleneck Problem**: The encoder compresses all contexts into a fixed 256D vector. Whether the context is 10 words or 70 words, the model has the same amount of information to work with. This creates a fundamental limitation.

2. **Empirical Evidence**: You should see ~30-40% accuracy degradation from short to long contexts:
   - Short contexts (8-12 words): ~88-95% accuracy
   - Medium contexts (25-35 words): ~75-80% accuracy  
   - Long contexts (50-70 words): ~50-60% accuracy

3. **Why Attention Matters**: The next lesson covers attention mechanisms, which allow the decoder to "look back" at the entire context during generation. Instead of compressing context into one vector, attention lets the model focus on relevant parts dynamically. This solves the bottleneck!

### Key Takeaways
- **Teacher Forcing**: Provides stable training by using ground truth tokens as input
- **Bidirectional Encoder**: Captures context from both directions
- **Context Vector**: The fixed-size bottleneck that limits model performance
- **Attention (Next)**: Dynamic context focusing to overcome the bottleneck

In Lesson 5, we'll add attention to eliminate this bottleneck and see performance improve significantly!